# Six-book debug fd_cache generator (Google Colab GPU)

**Mục đích**: Sinh trước **chỉ** các ký tự chu-Nom xuất hiện trong 6 bộ data
(CacThanhTruyen2/4/11 + SachThanhTruyen2/4/11) để chạy thử full pipeline
(`run_pipeline.sh`) end-to-end **trước**. Khi pipeline đã chạy đúng, bạn quay
lại notebook chính (`diffusion_colab.ipynb`) sinh full ~21.8k ký tự cho lần
chạy production cuối.

**Số ký tự cần sinh**: ~10.473 (union 6 sách, sau khi dedup) — bằng ~48% của
universe đầy đủ. Trên T4 ~5–7 h, trên L4 ~3–4 h, trên A100 ~1.5–2 h, trên
H100 ~45–60 phút.

**Hardware (chọn nhanh nhất bạn có)** — Settings → Runtime → Change runtime type:
1. **H100 GPU** ⭐ nhanh nhất (~3–4× T4) — yêu cầu compute units / Pro+
2. **A100 GPU** — yêu cầu compute units / Pro
3. **L4 GPU** — Pro
4. **T4 GPU** — free tier
5. ⚠️ Tránh **TPU (v5e/v6e)**: FontDiffusion chạy diffusers + CUDA-only ops, không port nguyên trạng sang TPU được.

**Resume-able**: Notebook đẩy state lên một HF Hub repo **riêng** cho debug
(`mdnt571/gannhanocr-debug-six`) để không trộn lẫn với cache production. Mở
lại notebook nếu Colab disconnect — cell 5 sẽ pull state về và bỏ qua các
ký tự đã có.

**Output**: `mdnt571/gannhanocr-debug-six` (dataset) chứa ~10.473 PNG `U+XXXX.png`. Final size ~750 MB.

**Style**: Cùng medoid crop với notebook chính để đảm bảo style đồng nhất giữa debug run và production run.


## 1. Verify GPU + install deps

Check that GPU is enabled (Runtime → Change runtime type). **Khuyến nghị H100 GPU**
để debug nhanh nhất; nếu không có compute units thì lần lượt A100 → L4 → T4.

In [5]:
!nvidia-smi || echo '⚠️  No GPU. Runtime → Change runtime type → H100/A100/L4/T4 GPU (theo thứ tự ưu tiên).'

import subprocess
gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip()
print(f'GPU detected: {gpu_name}')

# Colab base image often ships PyTorch already; we pin diffusers/HF stack.
# Don't downgrade torch unless its CUDA arch is wrong for the GPU.
%pip install -q \
    diffusers==0.36.0 \
    accelerate==1.12.0 \
    safetensors==0.7.0 \
    einops==0.8.1 \
    kornia==0.8.2 \
    info-nce-pytorch==0.1.4 \
    fonttools==4.61.1 \
    pygame==2.6.1 \
    scikit-image==0.26.0 \
    scipy==1.17.0 \
    huggingface-hub==0.36.0 \
    hf-xet==1.2.0 2>&1 | tail -3

# Drop peft/transformers — diffusers 0.36 imports a broken layerwise_casting helper from peft
# when peft is present, which crashes at model.to(). FontDiffusion doesn't need either.
%pip uninstall -y -q peft transformers 2>&1 | tail -2
print('✓ Removed peft/transformers (not needed by FontDiffusion).')

import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    supported = torch.cuda.get_arch_list()
    sm = f'sm_{cap[0]}{cap[1]}'
    ok = sm in supported
    print(f'  GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
    print(f'  Compute capability: {sm}  → {"✓ supported" if ok else "✗ NOT in supported list " + str(supported)}')
    if not ok:
        print('\n🛑 GPU not compatible with this PyTorch wheel. Switch to T4/L4/A100/H100.')

import diffusers
print(f'diffusers {diffusers.__version__}  (must be 0.36.0)')
assert diffusers.__version__ == '0.36.0', \
    f'diffusers version mismatch! Got {diffusers.__version__}, need 0.36.0. Restart runtime and re-run.'

try:
    import peft  # noqa: F401
    raise RuntimeError('peft is still installed — Runtime → Restart runtime and re-run this cell.')
except ImportError:
    print('✓ peft absent → diffusers will skip the broken layerwise_casting import.')


/bin/bash: line 1: nvidia-smi: command not found
⚠️  No GPU. Runtime → Change runtime type → H100/A100/L4/T4 GPU (theo thứ tự ưu tiên).


FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

## 2. Configure HuggingFace Hub

We push the partial cache to a HF Hub repo on a watcher cadence so a Colab disconnect không xoá tiến độ.

**Cách lấy `HF_TOKEN`**:
1. Tạo token (Write scope) tại https://huggingface.co/settings/tokens
2. Trên Colab: bấm icon chìa khoá (Secrets) ở thanh trái → **Add new secret** → Name = `HF_TOKEN`, Value = token, bật **Notebook access**.
3. Nếu không muốn dùng Secrets, bạn có thể paste thẳng vào `HF_TOKEN_FALLBACK` (kém an toàn hơn).

In [ ]:
import os
from huggingface_hub import HfApi, login, create_repo

# ════════════════════════════════════════════════════════════════════
# Edit HF_REPO to your dataset repo (auto-created if missing).
# Format: '<hf-username>/<repo-name>'
# ════════════════════════════════════════════════════════════════════
HF_REPO = 'mdnt571/gannhanocr'
HF_REPO_TYPE = 'dataset'
HF_TOKEN_FALLBACK = ''   # paste hf_xxx... here if not using Colab Secrets
# ════════════════════════════════════════════════════════════════════

HF_TOKEN = ''
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
    if HF_TOKEN:
        print('✓ HF_TOKEN loaded from Colab Secrets')
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = HF_TOKEN_FALLBACK or os.environ.get('HF_TOKEN', '')

assert HF_TOKEN, (
    'HF_TOKEN không tìm thấy. Mở tab Secrets bên trái Colab, thêm HF_TOKEN '
    '(Write scope) và bật Notebook access; hoặc paste vào HF_TOKEN_FALLBACK ở trên.'
)

login(token=HF_TOKEN, add_to_git_credential=False)
os.environ['HF_TOKEN'] = HF_TOKEN     # các thư viện con đọc env

api = HfApi()
me = api.whoami()
print(f'✓ Logged in as: {me["name"]}')

expected_user = HF_REPO.split('/')[0]
if me['name'] != expected_user:
    print(f'⚠️  Token belongs to "{me["name"]}" but HF_REPO uses "{expected_user}".')
    print(f'    Either change HF_REPO to "{me["name"]}/..." or use a token from "{expected_user}".')

try:
    create_repo(repo_id=HF_REPO, repo_type=HF_REPO_TYPE, exist_ok=True, private=False)
    print(f'✓ HF repo ready: https://huggingface.co/datasets/{HF_REPO}')
except Exception as e:
    print(f'⚠️  create_repo failed: {e}')
    print('   Token cần Write scope. Tạo token type "Write" thay cho Fine-grained nếu lỗi.')


## 2.5 Mount Google Drive (backup phụ song song với HF Hub)

**Đã bật mặc định** + đã point sang folder Drive bạn cung cấp:
`https://drive.google.com/drive/folders/1WrTyqvbLnaXXIwH_LtZMufbNcjN3l5NT`

Logic của cell:
1. Mount My Drive vào `/content/drive` — popup Google login lần đầu, các phiên sau auto reuse.
2. Gọi Drive API resolve folder ID → đường dẫn tuyệt đối dưới `/content/drive/MyDrive/...` (lần đầu sẽ có **popup auth thứ 2** cho Drive API; chấp nhận một lần thôi).
3. Mọi PNG sinh ra → song song lên HF Hub + folder Drive này.

**Yêu cầu**: account Colab phải có **quyền Editor** trên folder. Folder bạn share với chính mình → mặc định Editor.

In [ ]:
USE_DRIVE_BACKUP = True
DRIVE_BACKUP_FOLDER_ID = '1WrTyqvbLnaXXIwH_LtZMufbNcjN3l5NT'   # link bạn cung cấp
# Nếu sau này muốn đổi sang folder khác, paste ID mới (phần sau /folders/ trong URL).
# Để tắt Drive backup: USE_DRIVE_BACKUP = False

DRIVE_BACKUP_DIR = None
if USE_DRIVE_BACKUP:
    from google.colab import drive, auth as _ga_auth
    from pathlib import Path as _P

    # 1) Mount My Drive (FUSE) — popup Google login lần đầu
    drive.mount('/content/drive', force_remount=False)

    # 2) Auth Drive API (popup riêng lần đầu) để resolve folder ID → FUSE path
    _ga_auth.authenticate_user()
    from googleapiclient.discovery import build
    _svc = build('drive', 'v3')

    # 3) Walk parents từ folder ID để dựng đường dẫn tương đối dưới My Drive
    parts = []
    cur = DRIVE_BACKUP_FOLDER_ID
    for _ in range(20):  # depth guard
        meta = _svc.files().get(fileId=cur, fields='name,parents').execute()
        parents = meta.get('parents', [])
        if not parents:
            break       # đã đụng root (My Drive); không append tên root
        parts.append(meta['name'])
        cur = parents[0]
    parts.reverse()

    if not parts:
        raise RuntimeError(
            f'Folder ID {DRIVE_BACKUP_FOLDER_ID} resolve thành root — kiểm tra lại link.')

    DRIVE_BACKUP_DIR = _P('/content/drive/MyDrive', *parts)
    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

    # 4) Sanity write — fail sớm nếu account không có quyền Editor
    _probe = DRIVE_BACKUP_DIR / '.probe_write'
    try:
        _probe.write_text('ok')
        _probe.unlink()
    except Exception as e:
        raise RuntimeError(
            f'Không ghi được vào {DRIVE_BACKUP_DIR}: {e}\n'
            f'  → Account Colab cần quyền Editor trên folder.') from e

    print(f'✓ Drive backup dir: {DRIVE_BACKUP_DIR}')
    print(f'  (resolved from folder ID {DRIVE_BACKUP_FOLDER_ID})')
else:
    print('ℹ️  Drive backup disabled. HF Hub là kênh persistence duy nhất.')


## 3. Clone the GanNhanOCR repo (with submodule)

Pulls main repo + `font_diffusion` submodule (FontDiffusion code + fonts) vào `/content/GanNhanOCR`.

In [ ]:
import os, sys, shutil
from pathlib import Path

REPO_URL = 'https://github.com/truong571/GanNhanOCR.git'
PROJECT_ROOT = Path('/content/GanNhanOCR')

def _populated(root: Path) -> bool:
    return (root / 'font_diffusion' / 'fonts' / 'NomNaTong-Regular.ttf').exists()

if PROJECT_ROOT.exists():
    !cd {PROJECT_ROOT} && git pull --ff-only && git submodule update --init --recursive
    if not _populated(PROJECT_ROOT):
        shutil.rmtree(PROJECT_ROOT / 'font_diffusion', ignore_errors=True)
        !cd {PROJECT_ROOT} && git submodule update --init --recursive
else:
    !git clone --recurse-submodules --depth 1 {REPO_URL} {PROJECT_ROOT}

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

assert _populated(PROJECT_ROOT), 'Submodule clone failed — fonts not present.'
print(f'✓ Repo at {PROJECT_ROOT}')
!ls font_diffusion/fonts/ | head -5


In [ ]:
# ── 3.5  Materialize medoid.png inside the cloned repo ─────────────────────
#
# medoid.png is the cosine-medoid of 120 chu-Nom crops produced by
# kaggle_diffusion/build_style_medoid.py on the Mac. It is .gitignored
# (*.png blanket rule) so `git clone` in cell 3 does NOT bring it. We embed
# the file inline as base64 so the notebook is self-contained — no Kaggle
# Dataset, no manual upload, survives session reset.
#
# To refresh after re-running build_style_medoid.py on the Mac, regenerate the
# blob with:
#   python3 -c "import base64,textwrap; print(textwrap.fill(base64.b64encode(open('kaggle_diffusion/style_references/medoid.png','rb').read()).decode(), 76))"
# and paste the output between the triple-quoted MEDOID_B64 below.

import base64

MEDOID_B64 = """
iVBORw0KGgoAAAANSUhEUgAAAHQAAABZCAIAAABgypeYAAAFd0lEQVR4Ae3BAYqsSgIAwcz7HzoX
BEFpq0e7Lf8ybyKs+DOHFX/msOLPHFb8mcOKP3NY8WcOK/7MYcWfOaz4M4cVf+aw4s8cVsyksqj4
x1gxk8qq4l9ixUwqRyp+OyvmUzlS8XtZ8QiVIxW/lBXzqQxU/FJWPELlSMWt1Ir/A1Y8QuVIxX1U
jlQ8zopHqBypuInKWxUPsuIRKkcqbqLyVsWDrHiEypGKm6i8VfEgKx6hcqTiPipjFQ+yYj6VIxW3
UhmreJAVk6m8qJhAZaziQVZMpvKiYgKVsYoHWTGTypGKCVROq5jJislUXlTcTWWvUhmrmMaKyVT2
KiZQ2ahYqRypmMaKmVRWFTOprCr2VF5UTGPFNCqriitU9irGVDYqXqgcqZjAimlUVhVjKj+pGFNZ
VYypvKi4mxXTqKwqBlROqBhQ2agYUzlScSsrplFZVbxQOa3iiMpGxQkqAxV3sGIOlVXFEZVzKsZU
VhUnqAxU3MGKCVT2KhYqF1WMqWxUvFArNlTGKr5mxa1UBiqViyreUtmo2FNZVGyoDFR8zYqbqFxX
saeyqnhLZaPiiMqiYkPlrYpPWXETlYsqXqisKsZU9iqOqCwq9lTGKj5lxXVqxZ7KORVjKouKMZW9
igGVVcWeyljFR6y4SGVRsaeyUbFQWVW8pbKoGFPZqBhQ2ah4oTJQ8RErLlJZVZyjsqgYU1lVjKls
VIyp7FWMqawqPmLFFSp7FeeoQMWAykbFmMqq4i2VFxUDKhsV11lxkcpexR1UVhVjKnsVAypHKsZU
9iqusOI6lb2KO6gsKsZU9ipA5bSKt1T2Kk6z4iMqexXfUVlVDKhsVCxUTqv4icqLinOs+IjKXsV3
VF5UbKhsVGyonFBxjsqLihOs+JTKquInKouKIypHKlYqGxUnqGxUnKNypOInVnxKZaPiLZVFxU9U
VhUrlVXFOSobFeeoHKn4iRWfUtmoeEtlUfETlUXFSmWj4hyVjYoTVMYq3rLiUyp7FWMqi4ojKlCp
rCpWKquK01Q2Kn6i8pOKMSu+oLJXMaCyqNhTGahYqawqTlPZqBhQuajiiBVfUHlRcURlUbGhMlCx
UtmoOE1lr2JD5QsVL6z4gsqLiiMqq4qFykDFSmWj4gqVr1WAypGKPSu+oHKk4oXKaRUbKquK61S+
ULFSOVKxYcUXVI5UvFA5rWJPZVFxncoJFeeovKhYWfEdlSMVGyqnVdxE5ZyKj6isKlZWfEflCxX3
UTmn4j4qi4qVFV9Tua7iPiqnVdxNBSpWVtxB5a1KZVExoLKoOE3lior5rLiJykAFqCwqBlRWFeeo
XFExnxW3UnlRASqLigGVtyqOqCwqXqhsVMxnxd1U9ipAZVExoPJWxXUqGxXzWTGBykYFqCwqBlTe
qrhOZaNiPisepLKoOKLyVsV1KhsV81nxIJVFxRGVtyquU9momM+KB6ksKo6ovFVxncpGxXxWPEhl
UfFC5YSKi1Q2Kuaz4kEqi4o9ldMqrlDZqJjPimepLCpAZaACVI5UnKayV3FEBSqVRcVHrHiWyqIC
VI5UrFQGKk5Q+ULFdVY8SOUnFS9UflIxoPKFiuuseJDKWxVjKv+pitOseJDKQMU5Kt+p2FM5p+Ic
Kx6hMlZxkco5lcpGxQuVEyrOsWI+lbGK76i8qNhQWVWMqbxVcYIVE6gsKpWxijuorCpeqKwqxlTe
qjjBivuoXFTxCJVVxVsqYxUnWHETlRMq/gsqGxVvqRypOMeKm6j8pOI/orJR8ROVFxXnWHETlUWl
sqhUFhX/EZW9isms+Aeo7FVMZsU/QGWvYjIrfjuVvYr5rPjtVPYq5rPit1PZq5jPin+AyqriEVb8
G1QWFY+w4s8c/wOooDSL4xsCTAAAAABJRU5ErkJggg==
"""

style_dir = PROJECT_ROOT / 'kaggle_diffusion' / 'style_references'
style_dir.mkdir(parents=True, exist_ok=True)
medoid_path = style_dir / 'medoid.png'
medoid_path.write_bytes(base64.b64decode(MEDOID_B64))
print(f'✓ medoid.png materialized at {medoid_path}  ({medoid_path.stat().st_size} bytes)')


## 4. Download FontDiffusion checkpoints from HF Hub

Pulls the 3 production component files (unet, style_encoder, content_encoder).

In [ ]:
from huggingface_hub import hf_hub_download

FD_HF = 'dzungpham/font-diffusion-weights'
CKPT_DIR = PROJECT_ROOT / 'font_diffusion' / 'ckpt' / 'PROD'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

TO_FETCH = [
    'unet.safetensors',
    'style_encoder.safetensors',
    'content_encoder.safetensors',
]
for fn in TO_FETCH:
    dst = CKPT_DIR / fn
    if not dst.exists():
        cached = hf_hub_download(repo_id=FD_HF, filename=fn)
        shutil.copy2(cached, dst)
    print(f'  ✓ {fn}  ({dst.stat().st_size / 1024 / 1024:.1f} MB)')

FST_CKPT_DIR = CKPT_DIR  # use_fst=False → wrapper won't load FST modules
print(f'\n✓ FontDiffusion production weights ready at {CKPT_DIR}')


## 5. Resume from previous run (if any)

Pull các PNG đã có trên HF Hub về `/content/_universal_fd_cache`. Nếu mới lần đầu, snapshot rỗng — không sao.

In [ ]:
import os as _os
# Tắt progress bar per-file (tránh in hàng nghìn dòng 'U+XXXX.png: 100%')
_os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
from huggingface_hub import snapshot_download
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

CACHE_DIR = Path('/content/_universal_fd_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Pulling existing cache from {HF_REPO} ...')
try:
    snapshot_download(
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE,
        local_dir=str(CACHE_DIR),
        allow_patterns='U+*.png',
        max_workers=4,
        tqdm_class=None,        # tắt progress bar tổng
    )
except Exception as e:
    print(f'  (snapshot_download error: {type(e).__name__}: {str(e)[:200]})')

existing = sorted(CACHE_DIR.glob('U+*.png'))
print(f'✓ Resume state: {len(existing)} PNGs already local')


## 6. Build the work list (6-book union)

Đọc `kaggle_diffusion/exports/chars_six_union.txt` (~10.473 ký tự — union sau
dedup của 6 sách) thay vì full `char_universe.txt` (21.837 ký tự). Đây chính
là tập hợp tier-3 candidates mà `pipeline.step3_label` cần FontDiffusion
sinh ra cho 6 dataset hiện tại.

In [ ]:
universe_file = PROJECT_ROOT / 'kaggle_diffusion' / 'exports' / 'chars_six_union.txt'
assert universe_file.exists(), f'{universe_file} not found — đã commit kaggle_diffusion/exports/chars_six_union.txt vào repo chưa?'
with open(universe_file, encoding='utf-8') as f:
    all_chars = [line.strip() for line in f if line.strip()]

done_codepoints = {p.stem for p in existing}

todo = []
for c in all_chars:
    cp = f'U+{ord(c):04X}'
    if cp not in done_codepoints:
        todo.append(c)

print(f'Six-book union:  {len(all_chars):>6,} chars  (debug subset — full universe = 21,837)')
print(f'Already done:    {len(done_codepoints):>6,}')
print(f'To generate:     {len(todo):>6,} chars')
if not todo:
    print('\n🎉 Nothing to do — cache complete!')


## 7. Load FontDiffusion pipeline (one time)

In [ ]:
import logging, warnings
logging.getLogger('src.builders.build').setLevel(logging.ERROR)
logging.getLogger('diffusers').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

from core.ranking.fontdiffusion_gen import FontDiffusionGenerator

print('Loading FontDiffusion (~30 s)...')
generator = FontDiffusionGenerator(
    ckpt_dir=str(CKPT_DIR),
    phase1_ckpt_dir=str(FST_CKPT_DIR),
    font_path=str(PROJECT_ROOT / 'font_diffusion' / 'fonts' / 'NomNaTong-Regular.ttf'),
    cache_dir=str(CACHE_DIR),
)
print('✓ Loaded')


## 7.5 Sanity test — generate 5 sample chars

Trước khi commit vào run 6–8 h, sinh 5 ký tự đại diện và kiểm tra. Nhanh (~30 s) và bắt được:
- Style image hỏng / sai format
- FontDiffusion config mismatch
- GPU OOM
- Lỗi per-char

Nếu thumbnails trông giống chữ Nôm viết tay (không noise / blank), tiếp tục cell 8.

In [ ]:
import time, logging, warnings
import numpy as np
import cv2
from PIL import Image
from IPython.display import display

logging.getLogger('src.builders.build').setLevel(logging.ERROR)
logging.getLogger('diffusers').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

STYLE_NAME_ID = 'medoid'
GUIDANCE_SCALE = 2.0
ERODE_ITERS = 2

STYLE_IMAGE_PATH = str(PROJECT_ROOT / 'kaggle_diffusion' / 'style_references' / f'{STYLE_NAME_ID}.png')
assert Path(STYLE_IMAGE_PATH).exists(), f'Style image missing: {STYLE_IMAGE_PATH}'
print(f'Style: {STYLE_IMAGE_PATH}')
display(Image.open(STYLE_IMAGE_PATH).resize((128, 128)))

generator._load_pipeline()
generator.pipe.guidance_scale = GUIDANCE_SCALE
print(f'guidance_scale = {generator.pipe.guidance_scale}  |  erode_iters = {ERODE_ITERS}')


def erode_strokes(img_path, iters):
    try:
        arr = np.array(Image.open(img_path).convert('L'))
        if iters > 0:
            arr = cv2.dilate(arr, np.ones((3, 3), np.uint8), iterations=iters)
        Image.fromarray(arr).save(img_path)
    except Exception as e:
        print(f'  ⚠️  erode {img_path.name}: {type(e).__name__}: {e}')


TEST_CHARS = ['一', '二', '三', '人', '月']
print(f'\nGenerating {len(TEST_CHARS)} sanity-test chars: {TEST_CHARS}')

t0 = time.time()
generator.generate(TEST_CHARS, STYLE_IMAGE_PATH, style_name='_sanity')
print(f'  Time: {(time.time()-t0):.1f}s ({(time.time()-t0)/len(TEST_CHARS):.1f}s/char)')

sanity_dir = CACHE_DIR / '_sanity'
for c in TEST_CHARS:
    p = sanity_dir / f'U+{ord(c):04X}.png'
    if p.exists():
        erode_strokes(p, ERODE_ITERS)

print('\nGenerated images (post-erosion):')
for c in TEST_CHARS:
    img_path = sanity_dir / f'U+{ord(c):04X}.png'
    if img_path.exists():
        print(f'  ✓ {c} → U+{ord(c):04X}.png')
        display(Image.open(img_path).resize((96, 96)))
    else:
        print(f'  ✗ {c} → not generated')

import shutil as _sh
if sanity_dir.exists():
    _sh.rmtree(sanity_dir, ignore_errors=True)
print('\n✓ Sanity test complete. Inspect images above. If they look like handwritten chu-Nom, proceed.')


## 8. Generate + resilient checkpoint upload (Colab-tuned)

Vòng lặp: sinh theo chunk, watcher chạy nền flatten + push HF Hub mỗi khi đủ ~50 file mới hoặc ≥10 phút từ lần upload thành công gần nhất. Colab dễ disconnect hơn Kaggle nên `CHUNK` nhỏ hơn (200) và `WATCHER_INTERVAL_SEC=45` để giảm tổn thất.

Nếu bật `USE_DRIVE_BACKUP` ở cell 2.5, mỗi lần push HF thành công cũng rsync vào Drive như tầng dự phòng thứ hai.

In [6]:
import os, time, threading, traceback, logging, warnings, io, sys
from contextlib import redirect_stdout, redirect_stderr
import numpy as np
import cv2
from PIL import Image
from tqdm.auto import tqdm
import huggingface_hub
from huggingface_hub.utils import disable_progress_bars

# ── Quiet mode: tắt mọi progress bar + log không thiết yếu để giảm RAM/lag UI ──
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
disable_progress_bars()
logging.getLogger('src.builders.build').setLevel(logging.ERROR)
logging.getLogger('src.model').setLevel(logging.ERROR)
logging.getLogger('diffusers').setLevel(logging.ERROR)
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
logging.getLogger('huggingface_hub._upload_large_folder').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')
os.environ.setdefault('HF_XET_HIGH_PERFORMANCE', '1')

# Colab-tuned: chunk dày để giảm số commit (tránh 429 commit-per-hour limit)
CHUNK = 2000                       # ↑ từ 200 — chunk lớn → ít upload cycle
UPLOAD_NUM_WORKERS = 4
UPLOAD_RETRIES = 4
UPLOAD_BACKOFF_SECONDS = 60
WATCHER_INTERVAL_SEC = 60
WATCHER_MIN_FILES = 500            # ↑ từ 50 — chờ tích nhiều file rồi mới đẩy
MAX_UPLOAD_GAP_SEC = 1800          # ↑ từ 600 — 30 phút max gap
FILE_STABLE_AGE_SEC = 2
STYLE_NAME_ID = 'medoid'
GUIDANCE_SCALE = 2.0
ERODE_ITERS = 2
STYLE_NAME = 'universal'
STYLE_IMAGE = str(PROJECT_ROOT / 'kaggle_diffusion' / 'style_references' / f'{STYLE_NAME_ID}.png')

assert Path(STYLE_IMAGE).exists(), f'Style image missing: {STYLE_IMAGE}'
if not hasattr(api, 'upload_large_folder'):
    raise RuntimeError(
        f'huggingface_hub {huggingface_hub.__version__} lacks upload_large_folder(). '
        'Re-run cell 1, then Runtime → Restart runtime.'
    )

generator._load_pipeline()
generator.pipe.guidance_scale = GUIDANCE_SCALE
print(f'STYLE = {STYLE_NAME_ID} | guidance_scale = {GUIDANCE_SCALE} | erode_iters = {ERODE_ITERS}')
print(f'hf_hub = {huggingface_hub.__version__} | CHUNK = {CHUNK} | '
      f'watcher every {WATCHER_INTERVAL_SEC}s (≥{WATCHER_MIN_FILES} files OR ≥{MAX_UPLOAD_GAP_SEC}s)')
if DRIVE_BACKUP_DIR is not None:
    print(f'Drive backup → {DRIVE_BACKUP_DIR}')


def _erode_one(p):
    try:
        arr = np.array(Image.open(p).convert('L'))
        if ERODE_ITERS > 0:
            arr = cv2.dilate(arr, np.ones((3, 3), np.uint8), iterations=ERODE_ITERS)
        Image.fromarray(arr).save(p)
    except Exception:
        pass   # bỏ qua silently — file lỗi sẽ được retry phiên sau


sub = CACHE_DIR / STYLE_NAME
sub.mkdir(parents=True, exist_ok=True)

_flatten_lock = threading.Lock()
_devnull = io.StringIO()


def flatten(require_stable=True):
    moved = []
    now = time.time()
    with _flatten_lock:
        for p in list(sub.glob('U+*.png')):
            try:
                if require_stable and (now - p.stat().st_mtime) < FILE_STABLE_AGE_SEC:
                    continue
                dst = CACHE_DIR / p.name
                if not dst.exists():
                    _erode_one(p)
                    shutil.move(str(p), str(dst))
                    moved.append(dst)
                else:
                    p.unlink()
            except (FileNotFoundError, Exception):
                continue
    return moved


_upload_lock = threading.Lock()
_upload_state = {'thread': None, 'last_success': time.time()}


def _drive_snapshot():
    if DRIVE_BACKUP_DIR is None:
        return
    try:
        existing_drive = {p.name for p in DRIVE_BACKUP_DIR.glob('U+*.png')}
        new_files = [p for p in CACHE_DIR.glob('U+*.png') if p.name not in existing_drive]
        for p in new_files:
            shutil.copy2(p, DRIVE_BACKUP_DIR / p.name)
        if new_files:
            print(f'  💾 Drive: +{len(new_files):,}', flush=True)
    except Exception as e:
        print(f'  ⚠️  drive: {type(e).__name__}', flush=True)


def _upload_blocking(label):
    n = sum(1 for _ in CACHE_DIR.glob('U+*.png'))
    if n == 0:
        return True
    for attempt in range(1, UPLOAD_RETRIES + 1):
        try:
            # print_report=False → tắt 'Processing Files / New Data Upload' spam
            with redirect_stdout(_devnull):
                api.upload_large_folder(
                    repo_id=HF_REPO,
                    repo_type=HF_REPO_TYPE,
                    folder_path=str(CACHE_DIR),
                    allow_patterns='U+*.png',
                    num_workers=UPLOAD_NUM_WORKERS,
                    print_report=False,
                )
            _upload_state['last_success'] = time.time()
            print(f'  ✓ [{label}] pushed (HF={n:,})', flush=True)
            _drive_snapshot()
            return True
        except Exception as e:
            wait = UPLOAD_BACKOFF_SECONDS * attempt
            msg = str(e)[:120].replace(chr(10), ' ')
            print(f'  ⚠️  [{label}] try {attempt}/{UPLOAD_RETRIES}: {type(e).__name__}: {msg}', flush=True)
            if attempt == UPLOAD_RETRIES:
                print(f'  ⚠️  [{label}] giving up — local intact, retry next checkpoint.', flush=True)
                return False
            time.sleep(wait)
    return False


def _upload_target(label):
    with _upload_lock:
        try:
            _upload_blocking(label)
        except Exception as e:
            print(f'  ⚠️  [{label}] bg upload crashed: {type(e).__name__}', flush=True)


def maybe_upload_async(label):
    prev = _upload_state['thread']
    if prev is not None and prev.is_alive():
        return False
    t = threading.Thread(target=_upload_target, args=(label,), daemon=True)
    t.start()
    _upload_state['thread'] = t
    return True


def upload_wait():
    t = _upload_state['thread']
    if t is not None and t.is_alive():
        print('  ⏳ waiting for upload...', flush=True)
        t.join()


_watcher_state = {'stop': False, 'thread': None, 'pending': 0}


def _watcher_loop():
    while not _watcher_state['stop']:
        try:
            moved = flatten(require_stable=True)
            if moved:
                _watcher_state['pending'] += len(moved)
            prev = _upload_state['thread']
            idle = prev is None or not prev.is_alive()
            pending = _watcher_state['pending']
            gap = time.time() - _upload_state['last_success']
            if idle and pending > 0 and (pending >= WATCHER_MIN_FILES or gap >= MAX_UPLOAD_GAP_SEC):
                reason = f'+{pending}' if pending >= WATCHER_MIN_FILES else f'+{pending}/gap{int(gap)}s'
                _watcher_state['pending'] = 0
                maybe_upload_async(label=f'wch{reason}')
        except Exception:
            pass
        for _ in range(WATCHER_INTERVAL_SEC):
            if _watcher_state['stop']:
                return
            time.sleep(1)


def start_watcher():
    _watcher_state['stop'] = False
    _watcher_state['pending'] = 0
    _upload_state['last_success'] = time.time()
    t = threading.Thread(target=_watcher_loop, daemon=True)
    t.start()
    _watcher_state['thread'] = t
    print(f'  ▶  watcher started', flush=True)


def stop_watcher():
    _watcher_state['stop'] = True
    t = _watcher_state['thread']
    if t is not None:
        t.join(timeout=WATCHER_INTERVAL_SEC + 5)
    _watcher_state['thread'] = None


def safe_generate(batch):
    """generate.generate() in nhiều log per-batch — redirect_stdout để silence."""
    failed = []
    try:
        with redirect_stdout(_devnull):
            generator.generate(batch, STYLE_IMAGE, style_name=STYLE_NAME)
        return failed
    except Exception as e:
        msg = str(e)[:120].replace(chr(10), ' ')
        print(f'  ⚠️  batch gen failed: {type(e).__name__}: {msg} — fallback per-char', flush=True)
        torch.cuda.empty_cache()
        for c in batch:
            try:
                with redirect_stdout(_devnull):
                    generator.generate([c], STYLE_IMAGE, style_name=STYLE_NAME)
            except Exception as ce:
                failed.append((c, f'{type(ce).__name__}'))
                torch.cuda.empty_cache()
        if failed:
            print(f'  ⚠️  {len(failed)} chars failed in chunk', flush=True)
        return failed


if todo:
    t0 = time.time()
    # mininterval=10 → tqdm chỉ refresh mỗi 10s thay vì mỗi update → đỡ flood UI
    pbar = tqdm(total=len(todo), desc='Generate', unit='char',
                dynamic_ncols=True, mininterval=10.0)
    all_failed = []
    start_watcher()
    try:
        for i in range(0, len(todo), CHUNK):
            batch = todo[i:i + CHUNK]
            missing_batch = [c for c in batch if not (CACHE_DIR / f'U+{ord(c):04X}.png').exists()]

            if missing_batch:
                all_failed.extend(safe_generate(missing_batch))

            moved = flatten(require_stable=True)
            if moved:
                _watcher_state['pending'] += len(moved)

            pbar.update(len(batch))
            if maybe_upload_async(label=f'{min(i + CHUNK, len(todo)):,}/{len(todo):,}'):
                _watcher_state['pending'] = 0
            torch.cuda.empty_cache()
    except KeyboardInterrupt:
        print('\n⚠️  interrupted — letting in-flight upload finish.')
    except Exception as e:
        print(f'\n⚠️  loop error: {type(e).__name__}: {e}')
        traceback.print_exc()
    finally:
        pbar.close()
        stop_watcher()
        leftover = flatten(require_stable=False)
        if leftover:
            print(f'  flattened {len(leftover):,} leftover', flush=True)
        upload_wait()
        _upload_blocking(label='final')
        if all_failed:
            print(f'\n⚠️  {len(all_failed)} chars failed (first 5):')
            for c, err in all_failed[:5]:
                print(f'    {c} (U+{ord(c):04X}): {err}')
        print(f'\n✓ done in {(time.time()-t0)/60:.1f} min')
else:
    print('Nothing to generate.')


NameError: name 'PROJECT_ROOT' is not defined

## 9. Final state + verification

Sau khi run xong (hoặc session reset), kiểm tra cache khớp universe.

In [ ]:
final = sorted(CACHE_DIR.glob('U+*.png'))
size_mb = sum(p.stat().st_size for p in final) / 1024 / 1024
print(f'Final cache: {len(final):,} PNGs / {len(all_chars):,} target')
print(f'Total size: {size_mb:.0f} MB')
print(f'Coverage: {len(final)/len(all_chars)*100:.1f}%')

if len(final) < len(all_chars):
    missing_codepoints = set(f'U+{ord(c):04X}' for c in all_chars) - {p.stem for p in final}
    print(f'\n{len(missing_codepoints)} chars still missing.')
    print('Mở lại notebook để resume — cell 5 sẽ pull state từ HF và tiếp tục.')


## 10. Use the debug cache from your Mac

Khi đã 100%, ở Mac:

```sh
cd /path/to/GanNhanOCR
# pull debug cache (chỉ 6 sách) về một thư mục riêng
huggingface-cli download mdnt571/gannhanocr-debug-six \
  --repo-type=dataset \
  --local-dir prepared/_universal_fd_cache/

# chạy full pipeline end-to-end để debug
./run_pipeline.sh
```

`prepared/_universal_fd_cache/` chỉ chứa các ký tự cho 6 sách hiện tại — đủ
để step 3 (`pipeline.step3_label`) chạy không lỗi missing-glyph. Nếu thiếu
ký tự nào (vì step3 sinh chu-Nom mới ngoài union), step 3 sẽ render fallback
từ font.

**Sau khi pipeline đã chạy ổn**, quay lại `diffusion_colab.ipynb` (notebook
chính) để sinh full 21.837 ký tự cho production. Cache production push lên
`mdnt571/gannhanocr` (repo riêng) — không đè cache debug này.